# Module Template: Concept-to-Code Hands-On Lab
**Program:** 10-Day Rapid Hands-On AI/ML Engineering Bootcamp (GCP & Hybrid LLM)  
**Architecture:** Zero-Config Bootstrap | Local Hugging Face (`Qwen-Coder`) + GCP (`Vertex AI / Gemini`) Parity Router

### Cell 1 — Zero-Config Bootstrap (Read-Only)
Run this cell once. It silently validates pre-installed dependencies, detects GPU/CPU hardware, configures local Hugging Face cache paths (to avoid multi-GB downloads), and initializes the `ModelRouter`.

In [ ]:
import os
import json
import time
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Type
from pydantic import BaseModel

# Point Hugging Face to pre-cached shared disk/bucket mount if available
os.environ.setdefault("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
except ImportError:
    DEVICE = "cpu"

class ModelRouter:
    """
    Unified LLM Router enabling seamless switching between:
      1. 'local_hf': Local Hugging Face Qwen-Coder models (default, zero API cost, full privacy)
      2. 'vertex_gemini': GCP Vertex AI proprietary Gemini models via google-genai SDK
    """
    def __init__(
        self,
        provider: str = "local_hf",
        local_model: str = "Qwen/Qwen2.5-Coder-7B-Instruct",
        gcp_model: str = "gemini-2.5-flash",
        gcp_project: Optional[str] = None,
        gcp_location: str = "us-central1"
    ):
        self.provider = provider
        self.local_model_id = local_model
        self.gcp_model_id = gcp_model
        self.gcp_project = gcp_project or os.getenv("GOOGLE_CLOUD_PROJECT")
        self.gcp_location = gcp_location
        self._hf_pipeline = None
        self._genai_client = None
        print(f"[Bootstrap] Device detected: {DEVICE.upper()} | Active Provider: {self.provider}")

    def set_provider(self, provider: str):
        assert provider in ("local_hf", "vertex_gemini"), "Provider must be 'local_hf' or 'vertex_gemini'"
        self.provider = provider
        print(f"[ModelRouter] Switched active backend to: {self.provider}")

    def _init_local_hf(self):
        if self._hf_pipeline is not None:
            return
        from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
        print(f"[ModelRouter] Loading local Hugging Face model: {self.local_model_id}...")
        tokenizer = AutoTokenizer.from_pretrained(self.local_model_id)
        load_kwargs = {"device_map": "auto" if DEVICE != "cpu" else None}
        if DEVICE == "cuda":
            try:
                from transformers import BitsAndBytesConfig
                load_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True)
            except ImportError:
                load_kwargs["torch_dtype"] = "auto"
        model = AutoModelForCausalLM.from_pretrained(self.local_model_id, **load_kwargs)
        self._hf_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)

    def _init_vertex_gemini(self):
        if self._genai_client is not None:
            return
        from google import genai
        print(f"[ModelRouter] Initializing Vertex AI client ({self.gcp_model_id})...")
        self._genai_client = genai.Client(
            vertexai=True,
            project=self.gcp_project,
            location=self.gcp_location
        )

    def generate(
        self,
        prompt: str,
        system_instruction: str = "You are an expert AI engineering assistant.",
        temperature: float = 0.2,
        max_tokens: int = 512,
        response_schema: Optional[Type[BaseModel]] = None
    ) -> Dict[str, Any]:
        start_t = time.perf_counter()

        if self.provider == "local_hf":
            self._init_local_hf()
            sys_prompt = system_instruction
            if response_schema is not None:
                schema_json = json.dumps(response_schema.model_json_schema())
                sys_prompt += f"\nReturn ONLY valid JSON matching this schema: {schema_json}"

            messages = [
                {"role": "system", "content": sys_prompt},
                {"role": "user", "content": prompt}
            ]
            out = self._hf_pipeline(
                messages,
                max_new_tokens=max_tokens,
                temperature=max(temperature, 1e-4),
                do_sample=temperature > 0
            )
            text = out[0]["generated_text"][-1]["content"].strip()
            model_used = self.local_model_id

        else:
            self._init_vertex_gemini()
            from google.genai import types
            config_kwargs = {
                "system_instruction": system_instruction,
                "temperature": temperature,
                "max_output_tokens": max_tokens,
            }
            if response_schema is not None:
                config_kwargs["response_mime_type"] = "application/json"
                config_kwargs["response_schema"] = response_schema

            resp = self._genai_client.models.generate_content(
                model=self.gcp_model_id,
                contents=prompt,
                config=types.GenerateContentConfig(**config_kwargs)
            )
            text = resp.text.strip()
            model_used = self.gcp_model_id

        latency_ms = round((time.perf_counter() - start_t) * 1000, 2)
        return {
            "provider": self.provider,
            "model": model_used,
            "latency_ms": latency_ms,
            "output": text
        }

# Initialize router defaulting to local Hugging Face Qwen-Coder
llm = ModelRouter(provider="local_hf", local_model="Qwen/Qwen2.5-Coder-7B-Instruct")


### Cell 2 — Interactive Use-Case Selector (Anchor)
Start immediately with our built-in **Cloud FinOps & Incident Triage** enterprise scenario, or switch `USE_CASE_MODE = "CUSTOM"` to inject your own business problem and sample records.

In [ ]:
USE_CASE_MODE = "DEFAULT"  # Options: "DEFAULT" | "CUSTOM"

DEFAULT_USE_CASE = {
    "domain": "Cloud FinOps & SRE Incident Triage",
    "problem_statement": "Classify incoming production alerts, extract root-cause service metadata into strict JSON, and recommend automated remediation steps grounded in runbooks.",
    "sample_records": [
        "ALERT [P1]: gke-prod-payments podoomkilled in namespace checkout-api; memory limit 2Gi exceeded after v2.14.3 rollout.",
        "COST ANOMALY [P2]: BigQuery slot reservation spend spiked +340% in project analytics-prod due to unpartitioned table scan by svc-reporting."
    ],
    "knowledge_base": [
        "Runbook RB-101 (OOMKilled): Check recent deployment diff, inspect heap dump, and scale memory limit by 1.5x or rollback if leak confirmed.",
        "Runbook RB-204 (BigQuery Spend Spike): Enforce maximum_bytes_billed, partition tables by ingestion timestamp, and throttle svc-reporting quota."
    ]
}

# Optional: Populate if USE_CASE_MODE == "CUSTOM"
CUSTOM_USE_CASE = {
    "domain": "Your Domain Here (e.g., Retail Supply Chain / Healthcare Claims)",
    "problem_statement": "Describe the task you want the AI pipeline to solve...",
    "sample_records": [
        "Sample input record 1...",
        "Sample input record 2..."
    ],
    "knowledge_base": [
        "Domain policy or reference rule 1..."
    ]
}

ACTIVE_CASE = DEFAULT_USE_CASE if USE_CASE_MODE == "DEFAULT" else CUSTOM_USE_CASE
print(f"Active Use Case Loaded: {ACTIVE_CASE['domain']}")
print(f"Goal: {ACTIVE_CASE['problem_statement']}")


### Cell 3 — Micro-Concept & Architecture Visual (< 150 Words)
**Concept: Constrained Structured Extraction & Grounded Triage**  
Free-form LLM text breaks downstream microservices. By pairing **Pydantic JSON Schemas** with **Retrieval Grounding**, we force the model to output deterministic, type-safe payloads backed strictly by verified runbooks rather than parametric guesswork.

**When it breaks:** When schemas are deeply nested (>4 levels) on small quantized models (<7B) or when retrieved context exceeds the model's effective attention window.

```text
[Raw Incident/Record] ──► [Context Retriever] ──► [Prompt + Pydantic Schema]
                                                            │
                        ┌───────────────────────────────────┴───────────────────────────────────┐
                        ▼ (Default: Local HF)                                                   ▼ (Parity Toggle: GCP)
             [Qwen2.5-Coder-7B-Instruct]                                            [Vertex AI Gemini 2.5 Flash]
                        │                                                                       │
                        └───────────────────────────────────┬───────────────────────────────────┘
                                                            ▼
                                              [Validated JSON Output]
```

### Cell 4 — Guided Concept-to-Code (`# TODO: Candidate Implementation`)
Define the strict output schema and implement the `triage_record()` function using `llm.generate()`.

In [ ]:
from pydantic import Field

class TriageResult(BaseModel):
    severity: str = Field(description="Priority level such as P1, P2, or P3")
    affected_component: str = Field(description="Service, database, or entity impacted")
    root_cause_summary: str = Field(description="One-sentence root cause")
    recommended_action: str = Field(description="Action grounded strictly in the provided knowledge base")

def triage_record(record: str, knowledge_base: List[str]) -> Dict[str, Any]:
    """
    TODO: Candidate Implementation
    1. Construct a concise prompt combining `record` and `knowledge_base`.
    2. Call `llm.generate(...)` passing `response_schema=TriageResult`.
    3. Parse the returned JSON string into a Python dict and return alongside latency metrics.
    """
    kb_context = "\n".join(f"- {item}" for item in knowledge_base)
    prompt = (
        f"Analyze the following record using ONLY the reference knowledge base below.\n\n"
        f"Knowledge Base:\n{kb_context}\n\n"
        f"Record:\n{record}"
    )

    # Invoke active ModelRouter (defaults to local Hugging Face Qwen-Coder)
    response = llm.generate(
        prompt=prompt,
        system_instruction="Extract structured triage metadata accurately in JSON format.",
        temperature=0.1,
        response_schema=TriageResult
    )

    # Clean markdown fences if emitted by local open-weights model
    raw_json = response["output"].replace("```json", "").replace("```", "").strip()
    parsed = TriageResult.model_validate_json(raw_json)

    return {
        "provider": response["provider"],
        "model": response["model"],
        "latency_ms": response["latency_ms"],
        "structured_data": parsed.model_dump()
    }

# Execute on Sample Record 0 using Local Hugging Face Qwen-Coder
local_result = triage_record(ACTIVE_CASE["sample_records"][0], ACTIVE_CASE["knowledge_base"])
print(json.dumps(local_result, indent=2))


### Cell 5 — Proprietary Model Parity Toggle (GCP Vertex AI / Gemini)
Flip the provider flag to `"vertex_gemini"` and run the exact same function without changing a single line of business logic.

In [ ]:
# Toggle to proprietary GCP Vertex AI Gemini model
# Note: Requires GOOGLE_CLOUD_PROJECT env var & ADC configured in sandbox
RUN_GCP_PARITY_CHECK = False  # Set to True in GCP-connected sandbox

if RUN_GCP_PARITY_CHECK:
    llm.set_provider("vertex_gemini")
    gcp_result = triage_record(ACTIVE_CASE["sample_records"][0], ACTIVE_CASE["knowledge_base"])
    print(json.dumps(gcp_result, indent=2))
    # Reset back to local_hf for subsequent offline experiments
    llm.set_provider("local_hf")
else:
    print("Skipping live GCP call (RUN_GCP_PARITY_CHECK=False). Toggle to True when GCP project is attached.")


### Cell 6 — Automated Self-Check Gate
Validates that your implementation satisfies schema types, non-empty grounding, and latency constraints.

In [ ]:
def run_validation_gate(result: Dict[str, Any]):
    data = result.get("structured_data", {})
    assert isinstance(data, dict), "Output must be a valid dictionary"
    for key in ["severity", "affected_component", "root_cause_summary", "recommended_action"]:
        assert key in data and len(str(data[key]).strip()) > 0, f"Missing or empty field: {key}"
    print(f"[PASS] Schema & Grounding Gate passed on {result['model']} in {result['latency_ms']} ms!")

run_validation_gate(local_result)
